# Práctica 9 — Segmentación de clientes de una agencia automotriz\n## Aprendizaje no supervisado · K-Means · PCA · Generalización\n\n**Objetivo:** identificar grupos de clientes con características similares y analizar la complejidad y capacidad de generalización del modelo.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.cluster import KMeans\nfrom sklearn.decomposition import PCA\n

## 1. Datos de la práctica\nEl conjunto contiene 50 clientes y las 9 variables indicadas en el PDF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

datos = {
    "edad": [23,25,27,29,31,33,35,37,39,41,43,45,47,49,51,53,55,57,59,61,24,26,28,30,32,34,36,38,40,42,44,46,48,50,52,54,56,58,60,62,22,27,32,37,42,47,52,57,63,65],
    "ingreso_mensual": [12000,13500,14500,16000,18000,19500,21000,22500,24000,26000,28000,30000,32000,35000,38000,41000,45000,48000,52000,56000,13000,15000,17000,19000,21500,23500,25500,27500,29500,31500,33000,36000,39000,42000,46000,50000,54000,58000,62000,68000,11000,17500,23000,31000,37000,43000,49000,55000,65000,72000],
    "vehiculos_comprados": [1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,3,3,3,3,1,1,1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,3,3,4,1,1,2,2,2,3,3,3,4,4],
    "antiguedad_cliente": [1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,11,1,3,4,5,6,7,8,9,10,12],
    "gasto_servicio": [1500,1700,1800,2000,2200,2400,2600,2800,3000,3200,3500,3700,3900,4200,4500,4800,5200,5600,6000,6500,1600,1900,2100,2300,2500,2700,2900,3100,3400,3600,3800,4100,4400,4700,5000,5400,5800,6200,6700,7200,1400,2000,2700,3500,4300,5100,5900,6700,7500,8500],
    "visitas_taller": [1,1,1,2,2,2,2,3,3,3,3,4,4,4,5,5,5,6,6,7,1,1,2,2,2,3,3,3,4,4,4,5,5,5,6,6,6,7,7,8,1,2,3,4,5,6,7,8,9,10],
    "km_mensuales": [500,600,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,550,650,750,850,950,1050,1150,1250,1350,1450,1550,1650,1750,1850,1950,2050,2150,2250,2350,2500,450,800,1200,1600,2000,2400,2800,3200,3600,4000],
    "monto_compra": [180000,190000,200000,210000,220000,230000,240000,250000,260000,270000,280000,300000,320000,340000,360000,380000,400000,430000,460000,500000,185000,200000,215000,225000,235000,250000,265000,280000,295000,310000,330000,350000,370000,390000,420000,450000,480000,510000,550000,600000,175000,225000,275000,325000,375000,425000,475000,525000,575000,650000],
    "financiamiento": [1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0]
}

df = pd.DataFrame(datos)
columnas = list(df.columns)

\n

In [ ]:
print('Dimensiones:', df.shape)\nprint('\\nInformación:')\ndf.info()\nprint('\\nValores nulos por variable:')\nprint(df.isnull().sum())\ndisplay(df.describe())\n

### Respuestas — Exploración\n1. Hay **50 registros**.\n2. El conjunto contiene **9 variables**.\n3. **No existen valores nulos**.\n4. Variables económicas: ingreso_mensual, gasto_servicio, monto_compra y financiamiento.\n5. Variables de comportamiento: vehiculos_comprados, antiguedad_cliente, visitas_taller, km_mensuales y financiamiento.\n6. Las escalas son muy diferentes: monto_compra está en cientos de miles, ingreso en miles, edad en decenas y otras variables en unidades.

## 2. Preparación con StandardScaler\nK-Means utiliza distancias, por lo que trabajar con las escalas originales haría que variables de gran magnitud como monto_compra dominen el cálculo.

In [ ]:
scaler = StandardScaler()\nX = scaler.fit_transform(df)\nX = pd.DataFrame(X, columns=df.columns)\ndisplay(X.head())\n

## 3. Método del codo

In [ ]:
inercias = []\nfor k in range(2, 8):\n    modelo_k = KMeans(n_clusters=k, random_state=42, n_init=10)\n    modelo_k.fit(X)\n    inercias.append(modelo_k.inertia_)\n\nplt.figure(figsize=(8,5))\nplt.plot(range(2,8), inercias, marker='o')\nplt.xlabel('Número de clusters')\nplt.ylabel('Inercia')\nplt.title('Método del codo')\nplt.grid(alpha=0.25)\nplt.show()\n\npd.DataFrame({'K': range(2,8), 'Inercia': inercias})\n

### Elección de K\nSe considera razonable usar **3 clusters**: la caída de la inercia entre K=2 y K=3 es muy marcada y después la mejora continúa, pero con rendimientos menores. Esto mantiene un equilibrio entre simplicidad e interpretación.

## 4. K-Means con 3 clusters

In [ ]:
modelo = KMeans(n_clusters=3, random_state=42, n_init=10)\ndf_resultado = df.copy()\ndf_resultado['cluster'] = modelo.fit_predict(X)\n\nprint('Clientes por cluster:')\nprint(df_resultado['cluster'].value_counts().sort_index())\n\npromedios = df_resultado.groupby('cluster').mean(numeric_only=True)\ndisplay(promedios.round(2))\n

### Interpretación posterior al clustering\n- **Cluster 1:** valores promedio bajos de ingreso, compras, antigüedad y actividad; financiamiento frecuente.\n- **Cluster 2:** valores intermedios.\n- **Cluster 0:** valores promedio altos de ingreso, compra, antigüedad, gasto y actividad.\n\nLos números de cluster son identificadores, no niveles predefinidos.

## 5. PCA — reducción a dos dimensiones

In [ ]:
pca = PCA(n_components=2)\nX_pca = pca.fit_transform(X)\n\nprint('Varianza explicada por componente:', pca.explained_variance_ratio_)\nprint(f'Varianza total explicada: {pca.explained_variance_ratio_.sum()*100:.2f}%')\n\npca_df = pd.DataFrame(X_pca, columns=['Componente_1', 'Componente_2'])\npca_df['cluster'] = df_resultado['cluster'].values\ndisplay(pca_df.head())\n

In [ ]:
plt.figure(figsize=(9,6))\nsns.scatterplot(\n    data=pca_df, x='Componente_1', y='Componente_2',\n    hue='cluster', palette='deep', s=100\n)\nplt.title('Segmentación de clientes mediante PCA + K-Means')\nplt.grid(alpha=0.2)\nplt.show()\n

### Análisis PCA\n1. Los clusters muestran una separación general clara, aunque el segmento intermedio puede acercarse a los límites de otros grupos.\n2. Sí existen puntos cercanos entre clusters, especialmente en zonas de transición.\n3. Al reducir de 9 a 2 dimensiones se pierde parte del detalle de las otras componentes.\n4. Las dos primeras componentes explican aproximadamente **97.31 %** de la variabilidad.

## 6. Variables más importantes en PCA

In [ ]:
componentes = pd.DataFrame(\n    pca.components_,\n    columns=df.columns,\n    index=['Componente 1', 'Componente 2']\n)\ndisplay(componentes.round(3))\n\nprint('Mayores influencias absolutas en Componente 1:')\ndisplay(componentes.loc['Componente 1'].abs().sort_values(ascending=False))\nprint('Mayores influencias absolutas en Componente 2:')\ndisplay(componentes.loc['Componente 2'].abs().sort_values(ascending=False))\n

La Componente 1 está influida de forma similar por variables como gasto_servicio, ingreso_mensual, visitas_taller, monto_compra y edad. La Componente 2 está dominada por **financiamiento**, por lo que esta característica diferencia a los clientes en una dirección distinta al patrón general.

## 7. Complejidad del modelo y sobreajuste

In [ ]:
resultados = []\nfor k in range(2, 11):\n    modelo_k = KMeans(n_clusters=k, random_state=42, n_init=10)\n    modelo_k.fit(X)\n    resultados.append({'clusters': k, 'inercia': modelo_k.inertia_})\n\nresultados_df = pd.DataFrame(resultados)\ndisplay(resultados_df.round(2))\n\nplt.figure(figsize=(8,5))\nplt.plot(resultados_df['clusters'], resultados_df['inercia'], marker='o')\nplt.xlabel('Número de clusters')\nplt.ylabel('Inercia')\nplt.title('Complejidad del modelo')\nplt.grid(alpha=0.25)\nplt.show()\n

### Respuestas — Complejidad\n1. La inercia disminuye conforme aumenta K.\n2. Más clusters no significa automáticamente un mejor modelo.\n3. Si casi cada cliente queda en su propio grupo, la inercia puede ser mínima pero la segmentación deja de resumir patrones.\n4. Esto se relaciona con sobreajuste porque el modelo puede adaptarse a particularidades de la muestra.\n5. Una complejidad excesiva puede representar ruido en vez de patrones generales.

## 8. Generalización con clientes nuevos

In [ ]:
nuevos_clientes = pd.DataFrame({\n    'edad': [26, 35, 48, 55, 62],\n    'ingreso_mensual': [14000, 23000, 36000, 48000, 65000],\n    'vehiculos_comprados': [1, 1, 2, 3, 4],\n    'antiguedad_cliente': [1, 3, 6, 8, 11],\n    'gasto_servicio': [1800, 2500, 4200, 5600, 7800],\n    'visitas_taller': [1, 2, 4, 6, 9],\n    'km_mensuales': [600, 1000, 1700, 2200, 3500],\n    'monto_compra': [195000, 240000, 350000, 480000, 620000],\n    'financiamiento': [1, 1, 0, 0, 0]\n})\n\nnuevos_X = scaler.transform(nuevos_clientes)\nnuevos_clusters = modelo.predict(nuevos_X)\nnuevos_clientes['cluster_asignado'] = nuevos_clusters\ndisplay(nuevos_clientes)\n

### Respuestas — Generalización\n1. Sí, los cinco clientes nuevos pueden asignarse a los clusters existentes.\n2. Significa que la estructura aprendida puede aplicarse a observaciones que no participaron en el ajuste original.\n3. Debe utilizarse el mismo StandardScaler porque K-Means aprendió centroides en esa escala.\n4. Reentrenar el scaler solo con los clientes nuevos cambiaría la transformación y rompería la comparabilidad con los centroides originales.\n5. La asignación depende de la combinación de las nueve características; PCA muestra que muchas contribuyen conjuntamente y financiamiento destaca especialmente en la segunda componente.\n\nClusters esperados para los cinco clientes nuevos: **1, 1, 2, 0, 0**.

## 9. Sistema interactivo de segmentación\nLa siguiente interfaz permite capturar un cliente nuevo y demostrar la generalización del modelo.

In [ ]:
!pip -q install gradio\n

In [ ]:
import gradio as gr\n\npromedios_sistema = df_resultado.groupby('cluster').mean(numeric_only=True)\norden = promedios_sistema['ingreso_mensual'].sort_values().index.tolist()\nnombres_segmento = {\n    int(orden[0]): 'Segmento de entrada',\n    int(orden[1]): 'Segmento intermedio',\n    int(orden[2]): 'Segmento de alto valor',\n}\n\ndef interpretar_segmento(cluster):\n    nombre = nombres_segmento[int(cluster)]\n    if nombre == 'Segmento de entrada':\n        return nombre, 'Menor ingreso y compra promedio, poca antigüedad y mayor presencia de financiamiento.', 'Planes de financiamiento, servicio accesible y desarrollo de lealtad.'\n    if nombre == 'Segmento intermedio':\n        return nombre, 'Niveles medios de ingreso, compra, antigüedad y actividad.', 'Renovación de vehículo, paquetes de servicio y fidelización.'\n    return nombre, 'Mayores ingresos, compras, antigüedad, gasto en servicio y actividad.', 'Atención premium, mayor gama, servicio personalizado y retención.'\n\ndef segmentar_ui(edad, ingreso, vehiculos, antiguedad, gasto, visitas, km, monto, financiamiento):\n    cliente = pd.DataFrame([{\n        'edad': edad, 'ingreso_mensual': ingreso, 'vehiculos_comprados': vehiculos,\n        'antiguedad_cliente': antiguedad, 'gasto_servicio': gasto,\n        'visitas_taller': visitas, 'km_mensuales': km, 'monto_compra': monto,\n        'financiamiento': 1 if financiamiento == 'Sí' else 0\n    }], columns=df.columns)\n\n    cliente_x = scaler.transform(cliente)\n    cluster = int(modelo.predict(cliente_x)[0])\n    punto = pca.transform(cliente_x)[0]\n    nombre, detalle, recomendacion = interpretar_segmento(cluster)\n\n    fig, ax = plt.subplots(figsize=(8,5))\n    for c in sorted(pca_df['cluster'].unique()):\n        g = pca_df[pca_df['cluster'] == c]\n        ax.scatter(g['Componente_1'], g['Componente_2'], label=f'Cluster {c} · {nombres_segmento[int(c)]}', alpha=0.75)\n    ax.scatter(punto[0], punto[1], marker='*', s=280, edgecolors='black', label='Cliente nuevo')\n    ax.set_title('Cliente nuevo dentro de la segmentación PCA')\n    ax.set_xlabel('Componente 1')\n    ax.set_ylabel('Componente 2')\n    ax.grid(alpha=0.2)\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n\n    resultado = pd.DataFrame([{\n        'Cluster': cluster, 'Perfil': nombre,\n        'PC1': round(float(punto[0]), 3), 'PC2': round(float(punto[1]), 3)\n    }])\n    return f'Cluster {cluster}', nombre, detalle, recomendacion, resultado, fig\n\ntabla_clusters = promedios_sistema.round(2).reset_index()\ntabla_clusters.insert(1, 'perfil', [nombres_segmento[int(c)] for c in tabla_clusters['cluster']])\n\nwith gr.Blocks(title='AutoSegment AI') as sistema:\n    gr.Markdown('# AutoSegment AI\\n### Segmentación de clientes de una agencia automotriz')\n    with gr.Row():\n        with gr.Column():\n            edad_ui = gr.Number(value=48, label='Edad')\n            ingreso_ui = gr.Number(value=36000, label='Ingreso mensual ($)')\n            vehiculos_ui = gr.Number(value=2, label='Vehículos comprados')\n            antiguedad_ui = gr.Number(value=6, label='Antigüedad como cliente')\n            gasto_ui = gr.Number(value=4200, label='Gasto promedio en servicio ($)')\n        with gr.Column():\n            visitas_ui = gr.Number(value=4, label='Visitas al taller')\n            km_ui = gr.Number(value=1700, label='Kilómetros mensuales')\n            monto_ui = gr.Number(value=350000, label='Monto promedio de compra ($)')\n            financiamiento_ui = gr.Radio(['Sí','No'], value='No', label='Usa financiamiento')\n            boton_ui = gr.Button('Segmentar cliente', variant='primary')\n    with gr.Row():\n        cluster_ui = gr.Textbox(label='Cluster')\n        perfil_ui = gr.Textbox(label='Perfil comercial')\n    detalle_ui = gr.Textbox(label='Interpretación', lines=2)\n    recomendacion_ui = gr.Textbox(label='Recomendación ilustrativa', lines=2)\n    resultado_ui = gr.Dataframe(label='Resultado', interactive=False)\n    grafica_ui = gr.Plot(label='Visualización PCA')\n    with gr.Accordion('Promedios de los clusters', open=False):\n        gr.Dataframe(value=tabla_clusters, interactive=False)\n        gr.Markdown(f'Las dos componentes explican **{pca.explained_variance_ratio_.sum()*100:.2f}%** de la variabilidad.')\n\n    boton_ui.click(\n        fn=segmentar_ui,\n        inputs=[edad_ui, ingreso_ui, vehiculos_ui, antiguedad_ui, gasto_ui, visitas_ui, km_ui, monto_ui, financiamiento_ui],\n        outputs=[cluster_ui, perfil_ui, detalle_ui, recomendacion_ui, resultado_ui, grafica_ui]\n    )\n\nsistema.launch(share=True)\n

## Conclusión\nK-Means permitió descubrir tres segmentos sin utilizar etiquetas previas. StandardScaler fue necesario para comparar variables con escalas distintas y PCA conservó aproximadamente 97.31 % de la variabilidad en dos componentes. La prueba con clientes nuevos demuestra que el modelo puede aplicar la estructura aprendida a datos posteriores, siempre que se conserve la misma transformación.